In [0]:
# Provide access token (Note: Access token and Databricks url should be from same environment, ex: prod)
# dbutils.widgets.text("access_token", "", "access_token")
# access_token = dbutils.widgets.get("access_token")
access_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
print(f"access_token - {access_token}")


# Provide databricks url - Ex: https://adb-7627792522749697.17.azuredatabricks.net
dbutils.widgets.text("databricks_url", "", "databricks_url")
databricks_url = dbutils.widgets.get("databricks_url")
url = f"{databricks_url}/api/2.0/workspace/list"
print(f"url - {url}")

# Define the list of folder paths to search for matching words - Ex: /DATA_ENGINEERING, /deep_batch_framework/notebooks
dbutils.widgets.text("folder_paths", "", "folder_paths")
folder_paths = dbutils.widgets.get("folder_paths")
folder_paths = folder_paths.replace(" ", "")
folder_path_list = list(set(folder_paths.split(",")))
print(f"folder_path_list - {folder_path_list}")

# Define word list to search - Ex: t_d_xle_claim, t_d_xll_claim
dbutils.widgets.text("search_words", "", "search_words")
search_words = dbutils.widgets.get("search_words")
# search_words = search_words.replace(" ", "")
search_words_list = list(set(search_words.split(",")))
print(f"search_words_list - {search_words_list}")

access_token - [REDACTED]
url - https://adb-6031231472145779.19.azuredatabricks.net//api/2.0/workspace/list
folder_path_list - ['/Workspace/XLIIDW/XLIIDW_LOAD']
search_words_list - ['xliidw_uat_spl']


In [0]:
# Set the access token for the Databricks REST API
import requests
import base64
from requests.structures import CaseInsensitiveDict
import pandas as pd
import re
import os
os.environ['https_proxy'] = ''

# Define the request headers
headers = CaseInsensitiveDict()
headers["Authorization"] = f"Bearer {access_token}"
headers["Content-Type"] = "application/json"

# Define the request parameters as empty dictionary
params = {}

def read_contents(contents, path, folder_path):
    for item in contents:
        if item['object_type'] == 'DIRECTORY':
            # If item is a directory, it reads contents inside that directory
            new_path = item["path"]
            subdir_resp = requests.get(
                url=url,
                headers=headers,
                params={"path": new_path}
            )
            subdir_content = subdir_resp.json()
            read_contents(subdir_content.get('objects', []), new_path, folder_path)
        elif item['object_type'] == 'NOTEBOOK':
            # If item is a notebook, it checks its content for given words
            notebook_url = f"{databricks_url}/api/2.0/workspace/export"
            notebook_path = item['path']
            if notebook_path.startswith(folder_path):
                notebook_format = "SOURCE"
                notebook_params = {"path": notebook_path, "format": notebook_format}
                notebook_headers = CaseInsensitiveDict()
                notebook_headers["Authorization"] = f"Bearer {access_token}"
                notebook_headers["Content-Type"] = "application/json"
                notebook_resp = requests.get(notebook_url, headers=notebook_headers, params=notebook_params)
                decoded_base64_content = base64.b64decode(notebook_resp.json()['content']).decode('utf-8')
                matching_lines = []
                for word in search_words_list:
                    # To search word case-insensitive
                    lines = re.findall(fr"(?i).*\b{re.escape(word)}\b.*",decoded_base64_content) 
                    # To search word case-sensitive
                    # lines = re.findall(fr".*\b{re.escape(word)}\b.*",decoded_base64_content)
                    if lines:
                        matching_lines.append((word, lines))
                    else:
                        matching_lines.append((word, ''))
                if any(lines for _, lines in matching_lines):
                    notebook_contents.append((notebook_path, matching_lines))
                else:
                    notebook_contents.append((notebook_path, matching_lines))

# Read contents from all notebooks and subdirectories in specified folder paths
notebook_contents = []
for folder_path in folder_path_list:
    # Define the request parameters
    params = {"path": folder_path}

    # Send the request using the requests module
    resp = requests.get(url, headers=headers, params=params)
    data = resp.json()

    # Call the read_contents function to search for matching words
    read_contents(data['objects'], folder_path, folder_path)

# Create the pandas DataFrame from search results
df_list = []
for notebook_path, matching_lines in notebook_contents:
    word_present = ', '.join(word for word, lines in matching_lines if lines)
    lines_present = '\n\n'.join('\n'.join(lines) for word, lines in matching_lines if lines)
    if not word_present:
        word_present = ''
    if not lines_present:
        lines_present = ''
    df_list.append((notebook_path, word_present, lines_present))

df=spark.createDataFrame(df_list,['notebook_path', 'word_present', 'matching_lines'])

# Display the results as a table using Databricks display function
if df.rdd.isEmpty()==False: 
    display(df)
else:
    print("No matching notebooks found.")

---------------------------------------------------------------------------
PySparkNotImplementedError                Traceback (most recent call last)
File <command-1568132917310985>, line 84
     81 df=spark.createDataFrame(df_list,['notebook_path', 'word_present', 'matching_lines'])
     83 # Display the results as a table using Databricks display function
---> 84 if df.rdd.isEmpty()==False: 
     85     display(df)
     86 else:

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:2432, in DataFrame.rdd(self)
   2430 @property
   2431 def rdd(self) -> "RDD[Row]":
-> 2432     raise PySparkNotImplementedError(
   2433         error_class="NOT_IMPLEMENTED",
   2434         message_parameters={"feature": "rdd"},
   2435     )

PySparkNotImplementedError: [NOT_IMPLEMENTED] rdd is not implemented.

In [0]:
from pyspark.sql.functions import col

df.filter(col('word_present').isNotNull()).filter(col('word_present') != '').display()


notebook_path,word_present,matching_lines
/Workspace/XLIIDW/XLIIDW_LOAD/XLI_GENIUS/TransClaimOutwards,xliidw_uat_spl,"# ""param_src_db_name"":""xliidw_uat_spl.swiss_genius"", # ""param_tgt_db_name"":""xliidw_uat_spl.staging_genius"", # ""param_tbl_nme_genius_GWCLMTROT"":""xliidw_uat_spl.swiss_genius.GWCLMTROT"", # ""param_tbl_nme_staging_TransClaimOutwards"":""xliidw_uat_spl.staging_genius.TransClaimOutwards"", # ""param_tbl_nme_staging_T_Genius_Count"":""xliidw_uat_spl.staging_genius.T_Genius_Count"", # ""param_src_balance_sql"":""select count(1) as count from xliidw_uat_spl.swiss_genius.GWCLMTROT"", # ""param_tgt_balance_sql"":""select count(1) as count from xliidw_uat_spl.staging_genius.TransClaimOutwards"","
/Workspace/XLIIDW/XLIIDW_LOAD/XLI_GENIUS/AccruedPremium,xliidw_uat_spl,"# ""param_src_db_name"":""xliidw_uat_spl.staging_genius"", # ""param_tgt_db_name"":""xliidw_uat_spl.staging_genius"", # ""param_tbl_nme_genius_AccruedPremium_Incremental"":""xliidw_uat_spl.staging_genius.AccruedPremium_Incremental"", # ""param_tbl_nme_staging_AccruedPremium"":""xliidw_uat_spl.staging_genius.AccruedPremium"", # ""param_tbl_nme_staging_T_Genius_Count"":""xliidw_uat_spl.staging_genius.T_Genius_Count"", # ""param_src_balance_sql"":""select count(1) as count from xliidw_uat_spl.staging_genius.AccruedPremium_Incremental"", # ""param_tgt_balance_sql"":""select count(1) as count from xliidw_uat_spl.staging_genius.AccruedPremium"","
/Workspace/XLIIDW/XLIIDW_LOAD/XLI_GENIUS/AccruedPremium_Incremental,xliidw_uat_spl,"# ""param_src_db_name"":""xliidw_uat_spl.swiss_genius"", # ""param_tgt_db_name"":""xliidw_uat_spl.staging_genius"", # ""param_tbl_nme_genius_GWACPRMDP"":""xliidw_uat_spl.swiss_genius.GWACPRMDP"", # ""param_tbl_nme_staging_AccruedPremium_Incremental"":""xliidw_uat_spl.staging_genius.AccruedPremium_Incremental"", # ""param_tbl_nme_staging_T_Genius_Count"":""xliidw_uat_spl.staging_genius.T_Genius_Count"", # ""param_src_balance_sql"":""select count(1) as count from xliidw_uat_spl.swiss_genius.GWACPRMDP"", # ""param_tgt_balance_sql"":""select count(1) as count from xliidw_uat_spl.staging_genius.AccruedPremium_Incremental"","
/Workspace/XLIIDW/XLIIDW_LOAD/XLI_GENIUS/TransClaimInwards,xliidw_uat_spl,"# ""param_src_db_name"":""xliidw_uat_spl.swiss_genius"", # ""param_tgt_db_name"":""xliidw_uat_spl.staging_genius"", # ""param_tgt_tbl_name"":""xliidw_uat_spl.staging_genius.TransClaimInwards"", # ""param_tbl_nme_genius_GWCLMTRIN"":""xliidw_uat_spl.swiss_genius.GWCLMTRIN"", # ""param_tbl_nme_staging_TransClaimInwards"":""xliidw_uat_spl.staging_genius.TransClaimInwards"", # ""param_tbl_nme_staging_T_Genius_Count"":""xliidw_uat_spl.staging_genius.T_Genius_Count"", # ""param_src_balance_sql"":""select count(1) as count from xliidw_uat_spl.swiss_genius.GWCLMTRIN"", # ""param_tgt_balance_sql"":""select count(1) as count from xliidw_uat_spl.staging_genius.TransClaimInwards"","
/Workspace/XLIIDW/XLIIDW_LOAD/XLI_US_GENIUS/AccruedPremium,xliidw_uat_spl,"# ""param_src_db_name"":""xliidw_uat_spl.staging_genius"", # ""param_tgt_db_name"":""xliidw_uat_spl.staging_genius"", # ""param_tbl_nme_genius_AccruedPremium_Incremental"":""xliidw_uat_spl.staging_genius.AccruedPremium_Incremental"", # ""param_tbl_nme_staging_AccruedPremium"":""xliidw_uat_spl.staging_genius.AccruedPremium"", # ""param_tbl_nme_staging_T_Genius_Count"":""xliidw_uat_spl.staging_genius.T_Genius_Count"", # ""param_src_balance_sql"":""select count(1) as count from xliidw_uat_spl.staging_genius.AccruedPremium_Incremental"", # ""param_tgt_balance_sql"":""select count(1) as count from xliidw_uat_spl.staging_genius.AccruedPremium"","
/Workspace/XLIIDW/XLIIDW_LOAD/XLI_US_GENIUS/AccruedPremium_Incremental,xliidw_uat_spl,"# ""param_src_db_name"":""xliidw_uat_spl.swiss_genius"", # ""param_tgt_db_name"":""xliidw_uat_spl.staging_genius"", # ""param_tbl_nme_genius_GWACPRMDP"":""xliidw_uat_spl.swiss_genius.GWACPRMDP"", # ""param_tbl_nme_staging_AccruedPremium_Incremental"":""xliidw_uat_spl.staging_geni

In [0]:
# notebook_path_list = [f'/Workspace{row.notebook_path}' for row in df.select('notebook_path').distinct().collect()]
# print(f"""'{"', '".join(notebook_path_list)}'""")

---------------------------------------------------------------------------
PySparkNotImplementedError                Traceback (most recent call last)
File <command-7390131750120615>, line 84
     81 df=spark.createDataFrame(df_list,['notebook_path', 'word_present', 'matching_lines'])
     83 # Display the results as a table using Databricks display function
---> 84 if df.rdd.isEmpty()==False: 
     85     display(df)
     86 else:

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:2432, in DataFrame.rdd(self)
   2430 @property
   2431 def rdd(self) -> "RDD[Row]":
-> 2432     raise PySparkNotImplementedError(
   2433         error_class="NOT_IMPLEMENTED",
   2434         message_parameters={"feature": "rdd"},
   2435     )

PySparkNotImplementedError: [NOT_IMPLEMENTED] rdd is not implemented.

In [0]:
# spark.sql(f"""select distinct table_name from dw_xle_sz.t_axiom_table_lineage_raw_data where code_name in ('{"', '".join(notebook_path_list)}')""").display()

---------------------------------------------------------------------------
PySparkNotImplementedError                Traceback (most recent call last)
File <command-7390131750120615>, line 84
     81 df=spark.createDataFrame(df_list,['notebook_path', 'word_present', 'matching_lines'])
     83 # Display the results as a table using Databricks display function
---> 84 if df.rdd.isEmpty()==False: 
     85     display(df)
     86 else:

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:2432, in DataFrame.rdd(self)
   2430 @property
   2431 def rdd(self) -> "RDD[Row]":
-> 2432     raise PySparkNotImplementedError(
   2433         error_class="NOT_IMPLEMENTED",
   2434         message_parameters={"feature": "rdd"},
   2435     )

PySparkNotImplementedError: [NOT_IMPLEMENTED] rdd is not implemented.